---

## **Part 1: LangChain Framework – Core Architecture (Agentic Use Case Perspective)**

### **The Problem LangChain Solves**
LLMs (like GPT-4) are great at generating text, but:
- They can’t access tools like search, calculator, or your company data.
- They forget everything between messages unless you give them memory.
- They can't reason step-by-step or make decisions dynamically.

**LangChain solves this by acting as a bridge.**

---

### **LangChain Key Components (Agentic Perspective)**

| Component | What It Does | Real Example |
|----------|--------------|--------------|
| **LLM** | Connects to GPT-4, Claude, Mistral, etc. | "Write an email to HR" |
| **PromptTemplate** | Creates reusable prompt structures | "Write a tweet about {topic}" |
| **Tool** | External actions (Search, API call, DB) | Weather lookup, PDF reader |
| **Agent** | Decides *when* and *how* to use tools | "Hmm… to answer this, I need weather + location" |
| **Memory** | Stores past conversations | Multi-turn chatbots |
| **Retriever** | Retrieves documents from a knowledge base | Vector search in RAG |
| **Chain** | Links LLM calls together in steps | Extract from PDF → Generate summary → Send email |

---

### **Visual Architecture: Simple Agentic System**

```
[User Query]
     |
     v
[LangChain Agent] --- decides --->
     |                           |
     |                          [Tool 1: Weather API]
     |                          [Tool 2: Flight Search]
     |                          [Tool 3: Calculator]
     |
[Final Answer to User]
```


---



---

## **Problem Statement**

In many real-world scenarios, users seek contextual information about the current date and time as part of their natural language queries. For example:

* *“What is today’s date?”*
* *“Can you tell me the current time before I schedule a meeting?”*

While large language models (LLMs) can generate approximate or static answers, they are **not inherently aware of real-time system information**. As a result, they may provide outdated or fabricated responses when asked about the current date or time.

To ensure accuracy, an intelligent assistant must:

1. **Understand natural language queries referring to temporal context.**
2. **Automatically invoke tools that fetch real-time system date and time.**
3. **Integrate tool outputs into a conversational, human-friendly response.**

This functionality is crucial in **productivity tools, scheduling assistants, personal reminders, and business applications**, where real-time awareness significantly enhances user experience and reliability.

---

## **Objective**

Design and implement a **LangChain-powered agent** that integrates real-time utilities for date and time. The agent should:

* **Detect** when a user query involves current date or time.
* **Automatically call the appropriate tool** (`get_current_date` or `get_current_time`).
* **Leverage LangChain’s tool-calling framework** to handle reasoning and orchestration.
* **Maintain a reasoning trace** using `agent_scratchpad` for clarity.
* **Provide responses** in a conversational and context-aware format, ensuring outputs are both precise and user-friendly.

---



In [1]:
# !pip install -U \
# langchain==0.3.27 \
# langchain-core==0.3.79 \
# langchain-community==0.3.31 \
# langchain-openai==0.3.35 \
# langchain-text-splitters==0.3.9 \
# langchainhub \
# faiss-cpu \
# sentence-transformers -q

In [2]:
# Agent components:
# 1. Brain: LLM (e.g., GPT-4, Claude 3)
# 2. Prompt Template: Guides the LLM's responses.
# 3. Tools: Functions the agent can call to perform tasks.
# 4. Memory: Stores context from previous interactions.

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
# Langchain Simple Agent

# Import required libraries

# For calling the Openai API
from langchain_openai import ChatOpenAI # LLM BRAIN - ENGINE
# # For calling the Anthropic API
# from langchain_anthropic import ChatAnthropic # LLM BRAIN - ENGINE
# # For Azure OpenAI API
# from langchain_azure import ChatAzureOpenAI # LLM BRAIN - ENGINE
# # For calling the Google API
# from langchain_google import ChatGoogle # LLM BRAIN - ENGINE
# # For calling the Cohere API
# from langchain_cohere import ChatCohere # LLM BRAIN - ENGINE
# # For calling the Hugging Face API
# from langchain_huggingface import ChatHuggingFace # LLM BRAIN - ENGINE
# # For calling the Llama 2 API
# from langchain_llama import ChatLlama # LLM BRAIN - ENGINE
# # For calling the Gemini API
# from langchain_gemini import ChatGemini # LLM BRAIN - ENGINE
# # For calling the Mistral API
# from langchain_mistral import ChatMistral # LLM BRAIN - ENGINE

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # MESSAGES - CHASSIS
# ChatPromptTemplate: It is used to create a prompt template for chat models. This helps in structuring the input for the model.
# MessagesPlaceholder: It is used to indicate where the messages should be placed in the prompt template. This allows for dynamic insertion of messages during the conversation.

from langchain.tools import tool # HANDS - DASHBOARD
# tool: It is used to define a tool that can be called by the agent. This allows the agent to perform specific actions or retrieve information.

from langchain.agents import create_tool_calling_agent, AgentExecutor
# create_tool_calling_agent: It is used to create an agent that can call tools. This allows the agent to use the defined tools to perform tasks.
# AgentExecutor: It is used to execute the agent. This allows the agent to process inputs

In [5]:
import os
# os: It is used to interact with the operating system, such as setting environment variables.

from dotenv import load_dotenv
# load_dotenv: It is used to load environment variables from a .env file. This allows you to manage sensitive information like API keys without hardcoding them in your code.

In [6]:
# Step2: Load Environment Variables
load_dotenv()
# Load environment variables from a .env file
# This is useful for managing sensitive information like API keys without hardcoding them in your code.

True

In [7]:
llm = ChatOpenAI(model="gpt-4o")
# https://python.langchain.com/docs/integrations/chat/openai/

# ChatOpenAI(model, temperature, max_tokens, top_p, frequency_penalty, presence_penalty, stop, n, best_of, logit_bias, user, system_prompt, extra_kwargs)
# model: The name of the OpenAI model to use (e.g., "gpt-4o", "gpt-3.5-turbo").
# temperature: Controls the randomness of the output. Higher values (e.g., 0.8) make the output more random, while lower values (e.g., 0.2) make it more deterministic.
# max_tokens: The maximum number of tokens to generate in the response.
# top_p: Controls the diversity of the output by sampling from the top p% of the probability distribution.
# frequency_penalty: Controls how much to penalize new tokens based on their existing frequency in the text so far. Higher values (e.g., 0.5) make the model less likely to repeat the same lines verbatim.
# presence_penalty: Controls how much to penalize new tokens based on whether they appear in the text so far. Higher values (e.g., 0.5) make the model more likely to talk about new topics.
# stop: A list of tokens that will stop the generation when encountered.
# n: The number of responses to generate for each input message.
# best_of: The number of responses to generate and return the best (the one with the highest token logprobs per token).
# logit_bias: A dictionary that maps token IDs to bias values, which can be used to increase or decrease the likelihood of certain tokens appearing in the output.
# user: A string that can be used to identify the end-user, which can help OpenAI to monitor and detect abuse.
# system_prompt: A string that can be used to provide additional instructions or context to the model for the entire conversation.
# extra_kwargs: A dictionary of additional keyword arguments that can be passed to the OpenAI API.

In [8]:
# Step 5: Create prompt template (IMPORTANT: include input & agent_scratchpad)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a smart assistant. You can use tools whenever necessary to answer user questions."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # Required
    # MessagesPlaceholder - A special placeholder used by LangChain agents to store: a. Intermediate reasoning steps, b. Tool calls, c. Tool outputs - ReAct Style (Thoughts, Actions, and Observations)
])

In [9]:
# Demo Cell
from datetime import datetime, date, time
print(datetime.now())        # current date and time
# print(datetime.utcnow())     # UTC date and time
print(date.today())          # current date only
print(date.today().isoformat())  # current date in YYYY-MM-DD format
print(datetime.now().time())   # current time only
print(datetime.now().time().isoformat())  # current time in HH:MM:SS.microseconds format

2026-03-22 21:41:51.944762
2026-03-22
2026-03-22
21:41:51.945231
21:41:51.945256


In [10]:
# # Tool Definitions
# from datetime import datetime, date, time

# @tool
# def get_current_date():
#     """Returns the current date in YYYY-MM-DD format."""
#     return f"Current date is: {date.today().isoformat()}"

# @tool
# def get_current_time():
#     """Returns the current time in HH:MM:SS.microseconds format."""
#     return f"Current time is: {datetime.now().time().isoformat()}"

# tools = [get_current_date, get_current_time]

In [11]:
# !pip install pendulum -q
import pendulum

# Current UTC date-time (timezone-aware)
current_utc_datetime = pendulum.now("UTC")
print("Current UTC datetime:", current_utc_datetime.to_iso8601_string())

# Current UTC date only
current_utc_date = current_utc_datetime.date()
print("Current UTC date:", current_utc_date)

# Current UTC time only
current_utc_time = current_utc_datetime.time()
print("Current UTC time:", current_utc_time.isoformat())

Current UTC datetime: 2026-03-22T16:11:52.023335Z
Current UTC date: 2026-03-22
Current UTC time: 16:11:52.023335


In [17]:
import psutil

battery = psutil.sensors_battery()
print("Battery info:", battery)

if battery is None:
    print("Battery info not available on this system")
else:
    print(f"Battery: {battery.percent}%")
    print("Plugged in:", battery.power_plugged)
    if battery.secsleft not in (-1, -2):
        print("Seconds left:", battery.secsleft)

Battery info: sbattery(percent=24, secsleft=<BatteryTime.POWER_TIME_UNLIMITED: -2>, power_plugged=True)
Battery: 24%
Plugged in: True


In [18]:
# Tool Definitions
import pendulum

@tool
def get_current_utc_date_and_time():
    """Returns the current UTC date and time in ISO 8601 format."""
    current_utc_datetime = pendulum.now("UTC")
    return f"Current UTC date is: {current_utc_datetime.to_iso8601_string()} and current UTC time is: {current_utc_datetime.time().isoformat()}"

@tool
def get_battery_percentage():
    """Get the current battery percentage and charging status."""
    battery = psutil.sensors_battery()
    try:
        is_charging = battery.power_plugged # True if charging, False otherwise
        battery_percentage = battery.percent # Battery percentage
        if battery.percent is None: # Handle case where battery info is not available
            return "Battery information is not available."
        elif battery_percentage == 100 and is_charging:
            return "Battery is fully charged and plugged in. Please unplug the charger to prevent overcharging."
        elif battery_percentage == 100 and not is_charging:
            return "Battery is fully charged and not plugged in."
        elif battery_percentage > 80:
            return f"Battery is at {battery_percentage}% which is high. It is {'charging' if is_charging else 'not charging'}."
        elif battery_percentage > 60:
            return f"Battery is at {battery_percentage}% which is moderate. It is {'charging' if is_charging else 'not charging'}."
        elif battery_percentage > 40:
            return f"Battery is at {battery_percentage}% which is low. It is {'charging' if is_charging else 'not charging'}."
        elif battery_percentage > 20:
            return f"Battery is at {battery_percentage}% which is very low. It is {'charging' if is_charging else 'not charging'}. Please consider plugging in the charger soon."
        else:
            return f"Battery is critically low at {battery_percentage}%. It is {'charging' if is_charging else 'not charging'}. Please plug in the charger immediately to avoid shutdown."
    except Exception as e:
        return f"An error occurred while retrieving battery information: {str(e)}"


tools = [get_current_utc_date_and_time, get_battery_percentage]

In [19]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [20]:
# Wrap the agent in an AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # Set verbose=True to see the agent's reasoning and tool calls in the output
)

In [ ]:
response = agent_executor.invoke({"input": "Explain the process of photosynthesis in 5 lines."})
print(response['output'])
# print(response)

# How does the Agent work?
# 1. The user provides an input query to the agent.
# 2. The query "input" is passed to the agent's prompt template, which structures the input for the LLM.
# 3. This structured prompt is then sent to the LLM (e.g., GPT-4) for processing. LLM is good at understanding and generating human-like text based on the input it receives.
# 4. The LLM processes the input and looks for any tool calls if available. If the LLM determines that a tool is available, then it will call the tool with the appropriate parameters and get back the output from the tool. However, if the LLM determines that no tool is available, then it will see if it can answer the question itself based on its training data and the context provided in the prompt. If it can answer the question, it will generate a response directly. If it cannot answer the question, it will generate a response indicating that it cannot answer the question.
# 5. If the tool is called, the output from the tool is then passed back to the LLM as part of the agent's scratchpad (intermediate reasoning steps). The LLM can then use this information to generate a final response to the user's query.
# 6. The final response is then returned to the user. 



> Entering new AgentExecutor chain...
Photosynthesis is the process by which green plants, algae, and certain bacteria convert light energy into chemical energy in the form of glucose. It primarily occurs in the chloroplasts of plant cells, where chlorophyll absorbs sunlight. Carbon dioxide from the air and water from the soil are the reactants in this process. The light energy splits water molecules into oxygen and hydrogen, releasing oxygen as a byproduct. The hydrogen combines with carbon dioxide to form glucose, providing energy for the plant.

> Finished chain.
Photosynthesis is the process by which green plants, algae, and certain bacteria convert light energy into chemical energy in the form of glucose. It primarily occurs in the chloroplasts of plant cells, where chlorophyll absorbs sunlight. Carbon dioxide from the air and water from the soil are the reactants in this process. The light energy splits water molecules into oxygen and hydrogen, releasing oxygen as a byproduct. 

In [15]:
print(response)

{'input': 'Explain the process of photosynthesis in 5 lines.', 'output': 'Photosynthesis is the process by which green plants, algae, and certain bacteria convert light energy into chemical energy in the form of glucose. It primarily occurs in the chloroplasts of plant cells, where chlorophyll absorbs sunlight. Carbon dioxide from the air and water from the soil are the reactants in this process. The light energy splits water molecules into oxygen and hydrogen, releasing oxygen as a byproduct. The hydrogen combines with carbon dioxide to form glucose, providing energy for the plant.'}


In [16]:
response = agent_executor.invoke({"input": "Explain the process of photosynthesis in 5 bullet points."})
print(response['output'])




> Entering new AgentExecutor chain...
- **Light Absorption:** Chlorophyll, the green pigment in plant cells, captures sunlight, which provides the energy needed for photosynthesis.
- **Water Splitting:** Water molecules absorbed by the plant roots are split into oxygen, protons, and electrons through a process called photolysis.
- **Carbon Dioxide Fixation:** Carbon dioxide from the atmosphere is taken in by the plant leaves and is fixed into organic molecules in a series of reactions known as the Calvin Cycle.
- **Energy Conversion and Storage:** The energy from the sunlight is converted into chemical energy in the form of ATP and NADPH, which are used to convert carbon dioxide into glucose.
- **Oxygen Release:** As a byproduct of photolysis, oxygen is released into the atmosphere from the plant cells.

> Finished chain.
- **Light Absorption:** Chlorophyll, the green pigment in plant cells, captures sunlight, which provides the energy needed for photosynthesis.
- **Water Splitting:*

In [17]:
response = agent_executor.invoke({"input": "What is the capital of India?"})
print(response['output'])




> Entering new AgentExecutor chain...
The capital of India is New Delhi.

> Finished chain.
The capital of India is New Delhi.


In [18]:
response = agent_executor.invoke({"input": "What is the capital of New Delhi?"})
print(response['output'])




> Entering new AgentExecutor chain...
New Delhi is not a country or state that would have a capital; rather, it is itself the capital city of India.

> Finished chain.
New Delhi is not a country or state that would have a capital; rather, it is itself the capital city of India.


In [19]:
response = agent_executor.invoke({"input": "My name is Darshan and I want to become a data scientist. Please guide me with three successful career tips."})
print(response['output'])




> Entering new AgentExecutor chain...
Embarking on a career as a data scientist is an exciting and rewarding journey. Here are three tips to guide you on your path:

1. **Develop Strong Technical Skills**: 
   - **Programming:** Master programming languages such as Python and R, which are essential for data analysis and machine learning.
   - **Statistics and Mathematics:** A solid understanding of statistics, linear algebra, and calculus is crucial for developing models and interpreting data.
   - **Machine Learning and Data Analysis:** Gain proficiency in machine learning algorithms, data visualization, and data manipulation tools like Pandas, NumPy, and scikit-learn.

2. **Work on Real-world Projects**:
   - Engage in projects that require data collection, cleaning, analysis, and visualization. This practical experience is invaluable.
   - Participate in online competitions, such as Kaggle, to tackle real-world problems and learn from the community.
   - Build a portfolio to showc

In [20]:
response = agent_executor.invoke({"input": "What is the capital of New York?"})
print(response['output'])




> Entering new AgentExecutor chain...
The capital of New York is Albany.

> Finished chain.
The capital of New York is Albany.


In [21]:
response = agent_executor.invoke({"input": "Do do you recollect my name?"})
print(response['output'])




> Entering new AgentExecutor chain...
I’m sorry, but I don’t have the ability to remember personal details such as your name. Can you remind me?

> Finished chain.
I’m sorry, but I don’t have the ability to remember personal details such as your name. Can you remind me?


In [22]:
response = agent_executor.invoke({"input": "My name is Darshan and I want to become a data scientist. Please guide me with three successful career tips. At the end, give me a personalised tip, mentioning my name."})
print(response['output'])




> Entering new AgentExecutor chain...
Here are three successful career tips to guide you on your journey to becoming a data scientist:

1. **Build a Strong Foundation in Mathematics and Statistics:**
   Data science heavily relies on mathematical concepts and statistical analysis. Make sure to have a solid understanding of probability, statistics, linear algebra, and calculus. These fundamentals are crucial for data modeling and analysis.

2. **Master Programming Languages and Tools:**
   Familiarize yourself with programming languages commonly used in data science such as Python and R. Learn to work with data manipulation and analysis libraries like Pandas, NumPy, and tools for visualization like Matplotlib and Seaborn. Additionally, understanding SQL for database management is a must.

3. **Gain Practical Experience and Work on Real Projects:**
   Apply your skills by working on real-world data science projects. Participate in online competitions on platforms like Kaggle, contribut

In [23]:
response = agent_executor.invoke({"input": "Do do you recollect my name?"})
print(response['output'])




> Entering new AgentExecutor chain...
I don't have the ability to remember information between interactions, so I don't know your name. Could you please remind me?

> Finished chain.
I don't have the ability to remember information between interactions, so I don't know your name. Could you please remind me?


In [66]:
response = agent_executor.invoke({"input": "What is today's date in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:02:34.431562Z and current UTC time is: 16:02:34.431562The current UTC date and time is 2026-03-22T16:02:34. In India, the standard time zone is Indian Standard Time (IST), which is UTC+5:30.

So, the date in India would be:

- Date: 2026-03-22
- Time: 21:32 (UTC time + 5 hours 30 minutes)

Therefore, today's date in India is March 22, 2026.

> Finished chain.
The current UTC date and time is 2026-03-22T16:02:34. In India, the standard time zone is Indian Standard Time (IST), which is UTC+5:30.

So, the date in India would be:

- Date: 2026-03-22
- Time: 21:32 (UTC time + 5 hours 30 minutes)

Therefore, today's date in India is March 22, 2026.


In [67]:
response = agent_executor.invoke({"input": "What is today's date in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:02:52.404691Z and current UTC time is: 16:02:52.404691Today's date in India is March 22, 2026.

> Finished chain.
Today's date in India is March 22, 2026.


In [68]:
response = agent_executor.invoke({"input": "What is current time in India?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:03:12.891015Z and current UTC time is: 16:03:12.891015The current UTC time is 16:03. India is 5 hours and 30 minutes ahead of UTC, so the current time in India is 21:33.

> Finished chain.
The current UTC time is 16:03. India is 5 hours and 30 minutes ahead of UTC, so the current time in India is 21:33.


In [69]:
response = agent_executor.invoke({"input": "What is todays date and current time in India?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:03:30.465631Z and current UTC time is: 16:03:30.465631The current UTC date and time is 2026-03-22 at 16:03:30. India's time zone is UTC+5:30. Therefore, the current date and time in India is 2026-03-22 at 21:33:30.

> Finished chain.
The current UTC date and time is 2026-03-22 at 16:03:30. India's time zone is UTC+5:30. Therefore, the current date and time in India is 2026-03-22 at 21:33:30.


In [70]:
response = agent_executor.invoke({"input": "What is current time in 12hr format in India?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:03:50.080896Z and current UTC time is: 16:03:50.080896The current UTC time is 16:03, which corresponds to 9:33 PM in India's standard time (IST). Therefore, the current time in India in 12-hour format is 9:33 PM.

> Finished chain.
The current UTC time is 16:03, which corresponds to 9:33 PM in India's standard time (IST). Therefore, the current time in India in 12-hour format is 9:33 PM.


In [71]:
response = agent_executor.invoke({"input": "Is it 22nd March 2026 today in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:04:07.218255Z and current UTC time is: 16:04:07.218255Yes, it is 22nd March 2026 today in India, as it is currently 22nd March 2026 in UTC. India is ahead of UTC, so it would still be the same calendar day.

> Finished chain.
Yes, it is 22nd March 2026 today in India, as it is currently 22nd March 2026 in UTC. India is ahead of UTC, so it would still be the same calendar day.


In [72]:
response = agent_executor.invoke({"input": "Is it 22nd March 2026 today in Winnipeg, Canada?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:04:24.454957Z and current UTC time is: 16:04:24.454957Today is indeed 22nd March 2026 in Winnipeg, Canada, as it is currently 22nd March 2026 in UTC time.

> Finished chain.
Today is indeed 22nd March 2026 in Winnipeg, Canada, as it is currently 22nd March 2026 in UTC time.


In [73]:
response = agent_executor.invoke({"input": "Is it 22nd March 2026 today in NYC, USA?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:04:43.552025Z and current UTC time is: 16:04:43.552025Yes, today is 22nd March 2026 in UTC, which means it is also 22nd March 2026 in NYC, USA, since NYC is 4 or 5 hours behind UTC depending on daylight saving time.

> Finished chain.
Yes, today is 22nd March 2026 in UTC, which means it is also 22nd March 2026 in NYC, USA, since NYC is 4 or 5 hours behind UTC depending on daylight saving time.


In [ ]:
response = agent_executor.invoke({"input": "What is todays date and current time in NYC, USA?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:05:10.381944Z and current UTC time is: 16:05:10.381944The current UTC date and time is 2026-03-22 at 16:05:10. In New York City, which is in the Eastern Daylight Time (EDT) zone during this time of the year, the local time is 5 hours behind UTC. Therefore, the current date and time in NYC would be 2026-03-22 at 11:05:10.

> Finished chain.
The current UTC date and time is 2026-03-22 at 16:05:10. In New York City, which is in the Eastern Daylight Time (EDT) zone during this time of the year, the local time is 5 hours behind UTC. Therefore, the current date and time in NYC would be 2026-03-22 at 11:05:10.


In [75]:
response = agent_executor.invoke({"input": "What is todays date and current time in Winnipeg, Canada?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:05:48.753464Z and current UTC time is: 16:05:48.753464The current UTC date and time is 2026-03-22T16:05:48. In Winnipeg, Canada, which is in the Central Time Zone (CT), the standard time offset is UTC -6 hours, and during daylight saving time (DST), the offset is UTC -5 hours. Since today's date falls in March, it's likely that DST is in effect.

Therefore, the current date and time in Winnipeg is approximately 2026-03-22T11:05.

> Finished chain.
The current UTC date and time is 2026-03-22T16:05:48. In Winnipeg, Canada, which is in the Central Time Zone (CT), the standard time offset is UTC -6 hours, and during daylight saving time (DST), the offset is UTC -5 hours. Since today's date falls in March, it's likely that DST is in effect.

Therefore, the current date and time in Winnipeg is approximately 2026-03-22T11:05.


In [76]:
response = agent_executor.invoke({"input": "What is todays date and current time in Canbera, Australia?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:06:32.175957Z and current UTC time is: 16:06:32.175957The current UTC date and time is 2026-03-22T16:06:32.175957Z. Canberra, Australia is in the Australian Eastern Daylight Time (AEDT) timezone, which is 11 hours ahead of UTC. 

Therefore, the current date and time in Canberra is 2026-03-23 03:06:32 AEDT.

> Finished chain.
The current UTC date and time is 2026-03-22T16:06:32.175957Z. Canberra, Australia is in the Australian Eastern Daylight Time (AEDT) timezone, which is 11 hours ahead of UTC. 

Therefore, the current date and time in Canberra is 2026-03-23 03:06:32 AEDT.


In [77]:
response = agent_executor.invoke({"input": "Is it Sunday today in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:07:11.094997Z and current UTC time is: 16:07:11.094997The current UTC date and time is March 22, 2026, 16:07:11. India is 5 hours and 30 minutes ahead of UTC, so in India it is March 22, 2026, 21:37:11. March 22, 2026, is a Sunday. Therefore, it is Sunday today in India.

> Finished chain.
The current UTC date and time is March 22, 2026, 16:07:11. India is 5 hours and 30 minutes ahead of UTC, so in India it is March 22, 2026, 21:37:11. March 22, 2026, is a Sunday. Therefore, it is Sunday today in India.


In [50]:
response = agent_executor.invoke({"input": "On which date is India's Independence Day celebrated every year?"})
print(response['output'])




> Entering new AgentExecutor chain...
India's Independence Day is celebrated every year on August 15th.

> Finished chain.
India's Independence Day is celebrated every year on August 15th.


In [78]:
response = agent_executor.invoke({"input": "How many days have passed since the latest India's Republic Day?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:07:25.227510Z and current UTC time is: 16:07:25.227510India's Republic Day is celebrated on January 26th each year.

Given the current date is March 22, 2026, you can calculate the number of days since the latest Republic Day (January 26, 2026) by counting the days between these two dates.

1. January 26, 2026
2. February 26, 2026 (31 days from January 26 to February 26)
3. March 22, 2026 (26 + 22 = 48 days in total)

Therefore, 48 days have passed since the latest India's Republic Day as of March 22, 2026.

> Finished chain.
India's Republic Day is celebrated on January 26th each year.

Given the current date is March 22, 2026, you can calculate the number of days since the latest Republic Day (January 26, 2026) by counting the days between these two dates.

1. January 26, 2026
2. February 26, 2026 (31 days from January 26 to February 26)
3. March 22, 2026 

In [79]:
response = agent_executor.invoke({"input": "How many days have passed since the latest India's Independence Day? Explain step-by-step."})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:07:36.045279Z and current UTC time is: 16:07:36.045279India's Independence Day is celebrated on August 15th each year. To determine how many days have passed since the latest Independence Day, we need to follow these steps:

1. Identify the most recent Independence Day before the current date, which is August 15, 2025.
2. Calculate the difference in days between August 15, 2025, and the current date, March 22, 2026.

Now, let's calculate the number of days:

1. **Remaining days in August (2025)**: August 31 - August 15 = 16 days
2. **Full months between September 2025 and February 2026**:
   - September: 30 days
   - October: 31 days
   - November: 30 days
   - December: 31 days
   - January: 31 days
   - February 2026 (leap year): 29 days
3. **Days in March 2026 up to the 22nd**: 22 days

Adding these up:
- Days in August 2025: 16
- September 2025: 30
- Oct

In [80]:
response = agent_executor.invoke({"input": "my session is supposed to get over at 10 PM IST today. Can you tell me how much time I have teach more??"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:07:54.088214Z and current UTC time is: 16:07:54.088214The current UTC time is 16:07. India's time zone is UTC+5:30. Therefore, the current time in IST (Indian Standard Time) is 21:37.

Your session is supposed to end at 22:00 IST, so you have about 23 minutes left to teach.

> Finished chain.
The current UTC time is 16:07. India's time zone is UTC+5:30. Therefore, the current time in IST (Indian Standard Time) is 21:37.

Your session is supposed to end at 22:00 IST, so you have about 23 minutes left to teach.


In [81]:
response = agent_executor.invoke({"input": "What is current time in Pakistan?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:08:24.336796Z and current UTC time is: 16:08:24.336796Pakistan Standard Time (PST) is UTC+5 hours. Given the current UTC time is 16:08, the time in Pakistan would be 21:08.

> Finished chain.
Pakistan Standard Time (PST) is UTC+5 hours. Given the current UTC time is 16:08, the time in Pakistan would be 21:08.


In [82]:
response = agent_executor.invoke({"input": "What is the current time difference between IST and PST?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date_and_time` with `{}`


Current UTC date is: 2026-03-22T16:08:59.309524Z and current UTC time is: 16:08:59.309524India Standard Time (IST) is UTC+5:30, and Pacific Standard Time (PST) is UTC-8:00.

Currently, the UTC date and time is 2026-03-22T16:08:59. IST is 5 hours and 30 minutes ahead of UTC, and PST is 8 hours behind UTC. However, typically in March, daylight saving time in Pacific Time is observed, which means it likely shifts to Pacific Daylight Time (PDT), UTC-7:00.

If it's daylight saving time in effect for PST (which would be PDT, UTC-7:00), the difference would be:

**IST to PDT (UTC-7:00):** 12 hours and 30 minutes ahead

If daylight saving is not considered for any reason, the typical difference would be:

**IST to PST (UTC-8:00):** 13 hours and 30 minutes ahead

It's most likely that daylight saving theory applies, putting the difference at 12 hours and 30 minutes. If you want the exact time zone off

In [21]:
response = agent_executor.invoke({"input": "what is my current battery percentage of my laptop?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_battery_percentage` with `{}`


Battery is at 26% which is very low. It is charging. Please consider plugging in the charger soon.Your laptop's battery is currently at 26%, and it is charging.

> Finished chain.
Your laptop's battery is currently at 26%, and it is charging.


In [22]:
response = agent_executor.invoke({"input": "Is my laptop charging currently?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_battery_percentage` with `{}`


Battery is at 27% which is very low. It is charging. Please consider plugging in the charger soon.Yes, your laptop is currently charging. The battery is at 27%.

> Finished chain.
Yes, your laptop is currently charging. The battery is at 27%.


In [23]:
response = agent_executor.invoke({"input": "What is (2+3)?"})
print(response['output'])




> Entering new AgentExecutor chain...
The result of \(2 + 3\) is \(5\).

> Finished chain.
The result of \(2 + 3\) is \(5\).


The result of \(2 + 3\) is \(5\).

In [24]:
2+3

5

In [27]:
response = agent_executor.invoke({"input": "What is (5343*678)/23 + 45 - 67 + 89?"})
print(response['output'])




> Entering new AgentExecutor chain...
To calculate the expression \((5343 \times 678) / 23 + 45 - 67 + 89\), we need to follow the order of operations, often abbreviated as PEMDAS/BODMAS (Parentheses/Brackets, Exponents/Orders, Multiplication and Division (from left to right), Addition and Subtraction (from left to right)).

1. First, calculate the multiplication:
   \[
   5343 \times 678 = 3629274
   \]

2. Next, perform the division:
   \[
   3629274 / 23 \approx 157793.65
   \]

3. Then, proceed with the addition and subtraction:
   \[
   157793.65 + 45 = 157838.65
   \]
   \[
   157838.65 - 67 = 157771.65
   \]
   \[
   157771.65 + 89 = 157860.65
   \]

Therefore, the result of the expression is approximately \(157860.65\).

> Finished chain.
To calculate the expression \((5343 \times 678) / 23 + 45 - 67 + 89\), we need to follow the order of operations, often abbreviated as PEMDAS/BODMAS (Parentheses/Brackets, Exponents/Orders, Multiplication and Division (from left to right), A

The final result is approximately \(157,587.609\).

In [26]:
(5343*678)/23 + 45 - 67 + 89

157569.34782608695

In [ ]:
# Take Home Task:
# 1. Fix the calculator issue. Hint: You can use the Python's built-in eval() function to evaluate the mathematical expression. However, be cautious while using eval() as it can execute arbitrary code if the input is not properly sanitized. You can create a safe_eval function that only allows certain characters (digits, operators, parentheses) to be evaluated. Then, you can call this safe_eval function within your tool definition for the calculator.
# 2. Get the current CPU usage percentage. Hint: You can use the psutil library to get the current CPU usage percentage. The function psutil.cpu_percent() returns the current system-wide CPU utilization as a percentage. You can create a tool that calls this function and returns the CPU usage percentage when invoked by the agent.
# 3. Estimate tax based on annual income (India - simplified slab logic). Hint: You can create a tool that takes annual income as input and estimates the tax based on simplified slab logic for India. For example, you can define the following slabs:
# - Income up to ₹2.5 lakh: No tax
# - Income from ₹2.5 lakh to ₹5 lakh: 5% tax
# - Income from ₹5 lakh to ₹10 lakh: 20% tax
# - Income above ₹10 lakh: 30% tax

# Happy Learning